Task 1 - Install Spark, load required libraries, set environment variables, initiate Spark, load file

In [21]:
!apt-get install openjdk-8-jdk-headless -qq > /dev/null
!wget -q https://downloads.apache.org/spark/spark-3.5.0/spark-3.5.0-bin-hadoop3.tgz
!tar xf spark-3.5.0-bin-hadoop3.tgz
!pip install -q findspark

In [22]:
import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-8-openjdk-amd64"
os.environ["SPARK_HOME"] = "/content/spark-3.5.0-bin-hadoop3"
import findspark
findspark.init()
from google.colab import files
from pyspark.sql import SparkSession
from pyspark.ml.feature import VectorAssembler
from pyspark.sql.functions import isnan, when, count, col, lit
from pyspark.ml.regression import RandomForestRegressor
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.ml import Pipeline
from pyspark.ml.tuning import CrossValidator
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.ml.tuning import ParamGridBuilder


In [23]:
sc = SparkSession.builder.master("local[*]").getOrCreate()

In [ ]:
# Upload data.csv file
files.upload()

In [25]:
!ls

'data (1).csv'	 pipeline      spark-3.5.0-bin-hadoop3	     spark-3.5.0-bin-hadoop3.tgz.1
 data.csv	 sample_data   spark-3.5.0-bin-hadoop3.tgz


In [26]:
data = sc.read.csv('data.csv', inferSchema=True, header=True)

Task 2 - Describe and clean dataset

In [27]:
data.printSchema()
data.describe().toPandas().transpose()

root
 |-- Make: string (nullable = true)
 |-- Model: string (nullable = true)
 |-- Year: integer (nullable = true)
 |-- Engine Fuel Type: string (nullable = true)
 |-- Engine HP: integer (nullable = true)
 |-- Engine Cylinders: integer (nullable = true)
 |-- Transmission Type: string (nullable = true)
 |-- Driven_Wheels: string (nullable = true)
 |-- Number of Doors: integer (nullable = true)
 |-- Market Category: string (nullable = true)
 |-- Vehicle Size: string (nullable = true)
 |-- Vehicle Style: string (nullable = true)
 |-- highway MPG: integer (nullable = true)
 |-- city mpg: integer (nullable = true)
 |-- Popularity: integer (nullable = true)
 |-- MSRP: integer (nullable = true)



,0,1,2,3,4
summary,count,mean,stddev,min,max
Make,11914,None,None,Acura,Volvo
Model,11914,745.5822222222222,1490.8280590623795,1 Series,xD
Year,11914,2010.384337753903,7.5797398875957995,1990,2017
Engine Fuel Type,11911,None,None,diesel,regular unleaded
Engine HP,11845,249.38607007176023,109.19187025917194,55,1001
Engine Cylinders,11884,5.628828677213059,1.78055934824622,0,16
Transmission Type,11914,None,None,AUTOMATED_MANUAL,UNKNOWN
Driven_Wheels,11914,None,None,all wheel drive,rear wheel drive
Number of Doors,11908,3.4360933825999327,0.8813153865835529,2,4


In [28]:
def replace(column, value):
    return when(column != value, column).otherwise(lit(None))

data = data.withColumn("Market Category", replace(col("Market Category"), "N/A"))

In [29]:
data.select([count(when(isnan(c) | col(c).isNull(), c)).alias(c) for c in data.columns]).show()

+----+-----+----+----------------+---------+----------------+-----------------+-------------+---------------+---------------+------------+-------------+-----------+--------+----------+----+
|Make|Model|Year|Engine Fuel Type|Engine HP|Engine Cylinders|Transmission Type|Driven_Wheels|Number of Doors|Market Category|Vehicle Size|Vehicle Style|highway MPG|city mpg|Popularity|MSRP|
+----+-----+----+----------------+---------+----------------+-----------------+-------------+---------------+---------------+------------+-------------+-----------+--------+----------+----+
|   0|    0|   0|               3|       69|              30|                0|            0|              6|           3742|           0|            0|          0|       0|         0|   0|
+----+-----+----+----------------+---------+----------------+-----------------+-------------+---------------+---------------+------------+-------------+-----------+--------+----------+----+



In [30]:
data = data.drop('Market Category')
data = data.na.drop()
print((data.count(), len(data.columns)))

(11812, 15)


Part 3 - Create a Random Forest pipeline to predict car prices

In [31]:
data.columns

['Make',
 'Model',
 'Year',
 'Engine Fuel Type',
 'Engine HP',
 'Engine Cylinders',
 'Transmission Type',
 'Driven_Wheels',
 'Number of Doors',
 'Vehicle Size',
 'Vehicle Style',
 'highway MPG',
 'city mpg',
 'Popularity',
 'MSRP']

In [32]:
assembler = VectorAssembler(inputCols=['Year', 'Engine HP', 'Engine Cylinders',
                                       'Number of Doors', 'highway MPG', 'city mpg', 'Popularity'],
                            outputCol='Attributes')

regressor = RandomForestRegressor(featuresCol='Attributes', labelCol='MSRP')

pipeline = Pipeline(stages=[assembler, regressor])
pipeline.write().overwrite().save("pipeline")    # this ensures if we rerun the cell, it will overwrite the pipeline and not give any error

!ls

'data (1).csv'	 pipeline      spark-3.5.0-bin-hadoop3	     spark-3.5.0-bin-hadoop3.tgz.1
 data.csv	 sample_data   spark-3.5.0-bin-hadoop3.tgz


Part 4 - Create a cross validator for hyperparameter tuning

In [33]:
pipelineModel = Pipeline.load("pipeline")
paramGrid = ParamGridBuilder()\
              .addGrid(regressor.numTrees, [100,500])\
              .build()

crossval = CrossValidator(estimator = pipelineModel,
                          estimatorParamMaps = paramGrid,
                          evaluator = RegressionEvaluator(labelCol='MSRP'),
                          numFolds = 3)

Part 5 - Train your model and predict test set car prices

In [34]:
train_data, test_data = data.randomSplit([0.8, 0.2], seed = 123)

In [35]:
cvModel = crossval.fit(train_data)

In [36]:
bestModel = cvModel.bestModel
for x in range(len(bestModel.stages)):
    print(bestModel.stages[x])

VectorAssembler_e72585d11281
RandomForestRegressionModel: uid=RandomForestRegressor_bcc6e348569e, numTrees=500, numFeatures=7


In [37]:
pred = cvModel.transform(test_data)
pred.select('MSRP', 'prediction').show()

+-----+------------------+
| MSRP|        prediction|
+-----+------------------+
|28030|32794.495197985765|
|30550|37824.539035211325|
|29350|28368.675184881617|
|27900| 27668.42250420997|
|34890| 27668.42250420997|
|32990| 27668.42250420997|
| 2827| 4935.120021306275|
| 3000| 4935.120021306275|
| 3086| 4715.356546109408|
| 3130| 4715.356546109408|
| 3012|4954.8288437754145|
| 3622|  5985.22263633693|
|22300|24369.507516856724|
|19400|22533.453784703954|
| 2042| 4797.175180257896|
| 2144| 4813.267805153412|
|49440| 39803.86978149738|
|52640| 39803.86978149738|
|47440| 39959.49839149077|
|58400| 39949.77083558638|
+-----+------------------+
only showing top 20 rows



Part 6 - Evaluate your model's performance via several metrices

In [38]:
eval = RegressionEvaluator(labelCol='MSRP')
rmse = eval.evaluate(pred)    # default is rmse
mse = eval.evaluate(pred, {eval.metricName: "mse"})
mae = eval.evaluate(pred, {eval.metricName: "mae"})
r2 = eval.evaluate(pred, {eval.metricName: "r2"})

print("RMSE: %.3f" %rmse)
print("MSE: %.3f" %mse)
print("MAE: %.3f" %mae)
print("r2: %.3f" %r2)

RMSE: 36531.937
MSE: 1334582406.062
MAE: 9630.398
r2: 0.776
